Dataset => https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package?resource=download

# I/ Exploration

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder

DATASET_PATH = "./data/weatherAUS.csv"

In [2]:
def dataset_exploration(df):
    print(df.head())
    # Description des colonnes
    print(df.info())
    # Statisitiques
    print(df.describe())
    # Matrice de correlations
    plt.figure(figsize=(12, 8))
    sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt=".2f")
    plt.title("Corrélations entre les variables météo")
    plt.show()

In [3]:
df = pd.read_csv(DATASET_PATH)

# dataset_exploration(df)

In [4]:
# Pourcentage resultat vide par colonne
empty_value_percentage_df = (df.isnull().sum() / len(df)) * 100
print(empty_value_percentage_df)

Date              0.000000
Location          0.000000
MinTemp           1.020899
MaxTemp           0.866905
Rainfall          2.241853
Evaporation      43.166506
Sunshine         48.009762
WindGustDir       7.098859
WindGustSpeed     7.055548
WindDir9am        7.263853
WindDir3pm        2.906641
WindSpeed9am      1.214767
WindSpeed3pm      2.105046
Humidity9am       1.824557
Humidity3pm       3.098446
Pressure9am      10.356799
Pressure3pm      10.331363
Cloud9am         38.421559
Cloud3pm         40.807095
Temp9am           1.214767
Temp3pm           2.481094
RainToday         2.241853
RainTomorrow      2.245978
dtype: float64


# II/ Nettoyage

In [5]:
EMPTY_VALUE_PERCENTAGE_LIMITE = 30

In [6]:
# On supprime les colonnes ayant trop de valeur null
columns_to_drop_ls = empty_value_percentage_df[empty_value_percentage_df > EMPTY_VALUE_PERCENTAGE_LIMITE].index
print(f"Colonnes supprimées : {list(columns_to_drop_ls)}")
df = df.drop(columns=columns_to_drop_ls)

Colonnes supprimées : ['Evaporation', 'Sunshine', 'Cloud9am', 'Cloud3pm']


In [7]:
# On remplie les valeurs vide des colonnes restantes

# Valeur numeriques vide prenne la valeur mediane
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Valeur categoriel prenne la valeur la plus commune
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# III/ feature engineering

In [8]:
import numpy as np

In [9]:
# Extraction date
df['Date'] = pd.to_datetime(df['Date'])

# Extraction mois
df['Month'] = df['Date'].dt.month

# Écart de température
df['TempRange'] = df['MaxTemp'] - df['MinTemp']

# Moyenne d'humidité journalière
df['Humidity_Avg'] = (df['Humidity9am'] + df['Humidity3pm']) / 2

# Changement de pression
df['Pressure_Diff'] = df['Pressure3pm'] - df['Pressure9am']

In [10]:
# Conversion des directions en degrés
wd_map = {'N':0, 'NNE':22.5, 'NE':45, 'ENE':67.5, 'E':90, 'ESE':112.5, 'SE':135, 'SSE':157.5,
          'S':180, 'SSW':202.5, 'SW':225, 'WSW':247.5, 'W':270, 'WNW':292.5, 'NW':315, 'NNW':337.5}

df['WindDir3pm_deg'] = df['WindDir3pm'].map(wd_map)
df['WindDir3pm_sin'] = np.sin(np.deg2rad(df['WindDir3pm_deg']))
df['WindDir3pm_cos'] = np.cos(np.deg2rad(df['WindDir3pm_deg']))

# On supprime la colonne degre
df.drop('WindDir3pm_deg', axis=1, inplace=True)

In [11]:
# Tranforamtion Binaire
df['RainToday'] = df['RainToday'].map({'No': 0, 'Yes': 1})
df['RainTomorrow'] = df['RainTomorrow'].map({'No': 0, 'Yes': 1})

In [12]:
# Encodage Ville
le_city = LabelEncoder()
df['City_Encoded'] = le_city.fit_transform(df['Location'])

# Vision cyclique du temps => Pour pas que le model pense que janvier et l'opposé de decembre
df['Month_sin'] = np.sin(2 * np.pi * df['Date'].dt.month / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Date'].dt.month / 12)

In [13]:
# dataset_exploration(df)

In [14]:
df.to_csv('data/clean_data.csv', index=False)